In [26]:
import torch
import torch.nn.functional as F

from torch import Tensor

from transformers import AutoModel, AutoTokenizer
import numpy as np
from dataclasses import dataclass
import os
from typing import Any, Dict, Iterable, List, Optional, Tuple

In [27]:
# -------------------------
# Configuration
# -------------------------

@dataclass(frozen=True)
class Settings:
    # Elasticsearch
    es_host: str = os.getenv("ES_HOST", "http://127.0.0.1:9200")
    es_user: Optional[str] = os.getenv("ES_USER")
    es_password: Optional[str] = os.getenv("ES_PASSWORD")
    verify_certs: bool = os.getenv("ES_VERIFY_CERTS", "false").lower() == "true"
    index_name: str = os.getenv("ES_INDEX", "books_demo")

    # HF embedding model (auto-download)
    hf_model_id: str = "Qwen/Qwen3-Embedding-0.6B"
    hf_token: Optional[str] = os.getenv("HF_TOKEN")

    # If set, we will validate against model output dims
    embedding_dims: int = int(os.getenv("EMBED_DIMS", "1536"))

    # Embedding settings
    embedding_batch_size: int = int(os.getenv("EMBED_BATCH_SIZE", "16"))
    embedding_max_length: int = int(os.getenv("EMBED_MAX_LENGTH", "512"))
    device: str = os.getenv("DEVICE", "auto")  # auto | cpu | cuda | mps

In [28]:
from elasticsearch import Elasticsearch

def connect_es(cfg: Settings) -> Elasticsearch:
    kwargs = {
        "hosts": [cfg.es_host],
        "verify_certs": cfg.verify_certs,
        "request_timeout": 10,
    }
    if cfg.es_user and cfg.es_password:
        kwargs["basic_auth"] = (cfg.es_user, cfg.es_password)

    es = Elasticsearch(**kwargs)

    # Prefer info() over ping()
    info = es.info()
    print(f"[OK] Connected: {info['cluster_name']} (v{info['version']['number']}) @ {cfg.es_host}")
    return es

In [29]:
cfg = Settings()

In [30]:
es = connect_es(cfg)

[OK] Connected: docker-cluster (v8.11.3) @ http://127.0.0.1:9200


In [31]:
tokenizer = AutoTokenizer.from_pretrained(
    cfg.hf_model_id,
    padding_side="left",
    trust_remote_code=True,
    token=cfg.hf_token,
)

In [32]:
model = AutoModel.from_pretrained(
    cfg.hf_model_id,
    trust_remote_code=True,
    token=cfg.hf_token,
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 976.25it/s, Materializing param=norm.weight]                               


In [33]:
def pick_device(cfg: Settings) -> str:
    if cfg.device != "auto":
        return cfg.device
    try:
        import torch

        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

In [34]:
device = pick_device(cfg)
model = model.to(device)
model.eval()
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
# attention mask [[0,0,0,0,1,1], [0,0,0,1,1,1]]
# attention mask [[1,1,0,0,0,0], [1,1,1,0,0,0]]

In [54]:
def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

In [55]:
# def batch_embedding(tokenizer, model, texts, batch_size, max_length):
#     count = len(texts)
#     results = []
#     print(f"all Emebedding texts :{count} items")
#     for i in range(count//batch_size + 1):
#         print(f"Start to embedding:{i*batch_size}_{(i+1)*batch_size}")
#         input_texts = texts[i*batch_size:(i+1)*batch_size]
#         batch_dict = tokenizer(
#             input_texts,
#             padding=True,
#             truncation=True,
#             max_length = max_length,
#             return_tensors="pt"
#         )
#         batch_dict.to(model.device)
#         outputs = model(**batch_dict)
#         embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])

#         embeddings = F.normalize(embeddings, p=2, dim=1)

#         embeddings = embeddings.detach().cpu().float().numpy()
#         results.extend(embeddings)
#     return np.asarray(results)

import numpy as np
import torch
import torch.nn.functional as F

def batch_embedding(tokenizer, model, texts, batch_size, max_length):
    count = len(texts)
    results = []
    print(f"All embedding texts: {count} items")

    model.eval()
    with torch.no_grad():
        for start in range(0, count, batch_size):
            end = min(start + batch_size, count)
            print(f"Start embedding: {start}_{end}")

            input_texts = texts[start:end]
            batch_dict = tokenizer(
                input_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            batch_dict = {k: v.to(model.device) for k, v in batch_dict.items()}
            outputs = model(**batch_dict)

            embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)

            # -> numpy float32
            results.extend(embeddings.detach().cpu().float().numpy())

    return np.asarray(results, dtype=np.float32)


In [ ]:
import json
import random
from datetime import datetime, timedelta

TOPICS = [
    "Elasticsearch and Search Infrastructure",
    "Machine Learning Systems in Production",
    "Large Language Models and RAG",
    "Cloud Computing and Distributed Systems",
    "Data Engineering and Analytics",
    "AI Observability and Monitoring",
    "Recommendation Systems",
    "Information Retrieval and Ranking",
]

AUTHORS = [
    "Emily Chen",
    "Michael Zhang",
    "Sarah Johnson",
    "David Liu",
    "Rachel Kim",
    "Alex Martinez",
]

def random_date(start_year=2022):
    start = datetime(start_year, 1, 1)
    end = datetime.now()
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

def generate_article(idx: int) -> str:
    topic = random.choice(TOPICS)
    author = random.choice(AUTHORS)
    date = random_date().strftime("%B %d, %Y")

    paragraphs = [
        f"{topic}: Industry experts continue to explore new approaches to improving system performance, scalability, and reliability.",
        f"According to {author}, recent advances in this area have significantly reduced operational overhead while improving user experience.",
        "The report highlights challenges such as data quality, latency constraints, and cost optimization in large-scale deployments.",
        "Several organizations have adopted hybrid architectures combining traditional indexing techniques with modern vector-based retrieval methods.",
        "Looking ahead, researchers expect continued innovation driven by advances in hardware acceleration and foundation models.",
    ]

    body = "\n\n".join(paragraphs)

    return (
        f"{topic}\n\n"
        f"By {author} — {date}\n\n"
        f"{body}"
    )

# Write JSONL file
with open("doc1.json", "w", encoding="utf-8") as f:
    for i in range(1200):  # enough for your i > 1000 break
        record = {
            "docId": i,
            "contentText": generate_article(i),
            "source": "synthetic-news",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ doc1.json generated with synthetic English documents")


docs = []
with open("doc1.json", "r") as f:
    i = 0
    for line in f:
        docs.append(json.loads(line)["contentText"])
        if i>1000:
            break
        i+=1

✅ doc1.json generated with synthetic English documents


In [22]:
docs[0:2]

['Elasticsearch and Search Infrastructure\n\nBy Alex Martinez — February 03, 2022\n\nElasticsearch and Search Infrastructure: Industry experts continue to explore new approaches to improving system performance, scalability, and reliability.\n\nAccording to Alex Martinez, recent advances in this area have significantly reduced operational overhead while improving user experience.\n\nThe report highlights challenges such as data quality, latency constraints, and cost optimization in large-scale deployments.\n\nSeveral organizations have adopted hybrid architectures combining traditional indexing techniques with modern vector-based retrieval methods.\n\nLooking ahead, researchers expect continued innovation driven by advances in hardware acceleration and foundation models.',
 'AI Observability and Monitoring\n\nBy David Liu — August 20, 2025\n\nAI Observability and Monitoring: Industry experts continue to explore new approaches to improving system performance, scalability, and reliability

In [23]:
batch_embedding(tokenizer, model, docs, 16, 8192)

all Emebedding texts :1002 items
Start to embedding:0_16
Start to embedding:16_32
Start to embedding:32_48
Start to embedding:48_64
Start to embedding:64_80
Start to embedding:80_96
Start to embedding:96_112
Start to embedding:112_128
Start to embedding:128_144
Start to embedding:144_160
Start to embedding:160_176
Start to embedding:176_192
Start to embedding:192_208
Start to embedding:208_224
Start to embedding:224_240
Start to embedding:240_256
Start to embedding:256_272
Start to embedding:272_288
Start to embedding:288_304
Start to embedding:304_320
Start to embedding:320_336
Start to embedding:336_352
Start to embedding:352_368
Start to embedding:368_384
Start to embedding:384_400
Start to embedding:400_416
Start to embedding:416_432
Start to embedding:432_448
Start to embedding:448_464
Start to embedding:464_480
Start to embedding:480_496
Start to embedding:496_512
Start to embedding:512_528
Start to embedding:528_544
Start to embedding:544_560
Start to embedding:560_576
Start to 

array([[ 0.02575684,  0.02148438, -0.00592041, ..., -0.00531006,
         0.0267334 , -0.00476074],
       [-0.04418945,  0.00384521, -0.00793457, ...,  0.01611328,
         0.01495361, -0.0008316 ],
       [-0.01239014, -0.0189209 , -0.00567627, ...,  0.0402832 ,
         0.00085068, -0.00799561],
       ...,
       [-0.03759766, -0.01745605, -0.00668335, ..., -0.00382996,
        -0.00848389,  0.00976562],
       [ 0.02893066,  0.00848389, -0.00540161, ...,  0.00209045,
         0.02331543, -0.00430298],
       [-0.05102539, -0.04003906, -0.00289917, ...,  0.02429199,
         0.01104736,  0.0168457 ]], shape=(1002, 1024), dtype=float32)

### Search on should_list

In [43]:
dsl_kw = {
    "query": {
        "bool": {
            "must": [],
            "should": [],
            "must_not": [],
            "minimum_should_match": 0
        }
    },
    "size": 30
}

In [65]:
should_list = [
    {"match_phrase": {"title": {"query": "Hybrid Search", "boost": 2.0}}},
    {"terms": {"tags": ["vector-search", "elasticsearch"]}},
]

dsl_kw["query"]["bool"]["should"] = should_list

hits_kw = es.search(index="books_demo", body=dsl_kw)["hits"]["hits"]
print("num_hits =", len(hits_kw))
for h in hits_kw[:5]:
    print(h["_score"], h["_id"], h["_source"].get("title"))


num_hits = 1
12.745729 2 Hybrid Search: BM25 + Vector Retrieval


In [66]:
should_list = [
    {"match": {"title": {"query": "Hybrid Search", "boost": 2.0}}},
    {"terms": {"tags": ["vector-search", "elasticsearch"]}},
]

dsl_kw["query"]["bool"]["should"] = should_list

hits_kw = es.search(index="books_demo", body=dsl_kw)["hits"]["hits"]
print("num_hits =", len(hits_kw))
for h in hits_kw[:5]:
    print(h["_score"], h["_id"], h["_source"].get("title"))

num_hits = 1
12.745729 2 Hybrid Search: BM25 + Vector Retrieval


### Search on must_list

In [63]:
dsl_kw = {
    "query": {
        "bool": {
            "must": [
                {"match_phrase": {"title": {"query": "Hybrid Search", "boost": 2.0}}}
            ],
            "should": [
                {"match": {"title": {"query": "Vector", "boost": 2.0}}},
                {"terms": {"tags.keyword": ["vector-search", "elasticsearch"]}},
            ],
            "minimum_should_match": 0
        }
    },
    "size": 10,
    "_source": {"excludes": ["docEmbedding"]}
}

res_kw = es.search(index="books_demo", body=dsl_kw)
hits_kw = res_kw["hits"]["hits"]

print("num_hits =", len(hits_kw))
for h in hits_kw[:5]:
    print(h["_score"], h["_id"], h["_source"].get("title"))

num_hits = 1
8.622186 2 Hybrid Search: BM25 + Vector Retrieval


### Search for Multi Fields

In [62]:
dsl = {
  "query": {
    "multi_match": {
      "query": "Elasticsearch",
      "fields": ["title^2", "author"]
    }
  },
  "size": 10
}

dsl1 = {**dsl, "size": 5}   # override
res = es.search(index="books_demo", body=dsl1)

for h in res["hits"]["hits"]:
    print(h["_score"], h["_id"], h["_source"].get("title"))

1.3862942 3 Vector Databases vs Elasticsearch for RAG
1.2978076 4 Scaling Elasticsearch Clusters: Shards, Replicas, and Nodes
1.2199391 7 Observability Logs in Elasticsearch: Index Lifecycle and Tiers


In [60]:
# 1) Does the index have docs?
count_res = es.count(index="books_demo")
print("doc_count =", count_res["count"])

# 2) What fields exist?
props = es.indices.get_mapping(index="books_demo")["books_demo"]["mappings"]["properties"]
print("fields =", list(props.keys()))

# 3) Try a guaranteed hit
res = es.search(
    index="books_demo",
    body={"query": {"match_all": {}}, "size": 1}
)

for h in res["hits"]["hits"]:
    print(h["_score"], h["_id"], h["_source"].get("title"))

doc_count = 7
fields = ['author', 'docEmbedding', 'published_date', 'tags', 'title', 'views']
1.0 1 Lucene Internals: Inverted Index and Doc Values


### Search on Embedding

In [56]:
query_text = "Introduction to Elasticsearch"
query_vec = batch_embedding(
    tokenizer, model,
    [query_text],
    batch_size=1,
    max_length=cfg.embedding_max_length
)[0].tolist()


All embedding texts: 1 items
Start embedding: 0_1


In [58]:
dsl = {
    "knn": {
        "field": "docEmbedding",
        "query_vector": query_vec,
        "k": 10,
        "num_candidates": 100
    },
    "size": 10,
    "_source": ["title", "author", "published_date", "tags", "views"]  # exclude docEmbedding
}

hits = es.search(index="books_demo", body=dsl)["hits"]["hits"]
for h in hits:
    print(h["_score"], h["_id"], h["_source"]["title"])


0.79634094 7 Observability Logs in Elasticsearch: Index Lifecycle and Tiers
0.7930902 4 Scaling Elasticsearch Clusters: Shards, Replicas, and Nodes
0.7775297 1 Lucene Internals: Inverted Index and Doc Values
0.77452815 3 Vector Databases vs Elasticsearch for RAG
0.7505451 5 Practical Aggregations: Facets for E-commerce Search
0.7461879 6 Search Relevance Tuning with Query DSL
0.7387519 2 Hybrid Search: BM25 + Vector Retrieval


### Delete Doc

In [19]:
search_res = es.search(index="books_demo", query={"match_all": {}})

total_docs = search_res["hits"]["total"]["value"]
print(total_docs)

8


In [20]:
res = es.get(index="books_demo", id="0")
print(res["_source"])

{'title': 'Elasticsearch Basics', 'author': 'Emily', 'published_date': '2024-01-01', 'tags': ['search', 'elasticsearch'], 'views': 1024, 'docEmbedding': [0.0012359619140625, 0.05224609375, -0.005706787109375, -0.07080078125, -0.031494140625, -0.009765625, 0.048828125, -0.01202392578125, -0.0002498626708984375, -0.0260009765625, -0.048583984375, -0.037109375, 0.041015625, -0.0032806396484375, -0.025146484375, 0.10498046875, -0.07763671875, 0.07958984375, 0.0108642578125, -0.046142578125, -0.08349609375, 0.062255859375, 0.007293701171875, 0.04248046875, -0.025390625, -0.038818359375, 0.01300048828125, -0.00055694580078125, 0.0240478515625, -0.0517578125, 0.0380859375, -0.02685546875, -0.048583984375, -0.0164794921875, -0.0263671875, -0.0079345703125, -0.0185546875, 0.002044677734375, 0.041015625, -0.0140380859375, -0.006317138671875, 0.00109100341796875, 0.1162109375, 0.0184326171875, 0.0732421875, -0.07763671875, 0.0091552734375, 0.0079345703125, 0.0018157958984375, 0.0250244140625, -0.

In [22]:
from elasticsearch import NotFoundError

try:
    es.delete(index="books_demo", id="0")
    print("Deleted successfully")
except NotFoundError:
    print("Document not found")

Deleted successfully


In [24]:
try:
    res = es.get(index="books_demo", id="0")
    print(res["_source"])
except NotFoundError:
    print("Document not found")


Document not found
